In [1]:
import pandas as pd

import meteostat as ms
from datetime import datetime
from pathlib import Path
import gridstatus

In [2]:
start = datetime(2024, 1, 1)
end = datetime(2025, 12, 31, 23, 59)

points = {
    "la": ms.Point(34.0522, -118.2437),
    "sf": ms.Point(37.7749, -122.4194),
    "sd": ms.Point(32.7157, -117.1611),
    "sj": ms.Point(37.3382, -121.8863),   
    "fresno": ms.Point(36.7378, -119.7871) 
}

weather_dfs = {}

for name, point in points.items():
    stations = ms.stations.nearby(point, limit=4)
    ts = ms.hourly(stations, start, end)
    df = ms.interpolate(ts, point).fetch().reset_index()
    df = df.rename(columns={"time": "datetime"})
    weather_dfs[name] = df
    
    print(name, df.shape)
    print(df.head(2))


la (17544, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2024-01-01 00:00:00  15.9    68   0.0   117   8.7  <NA>  1017.2     8     3
1 2024-01-01 01:00:00  14.2    74   0.0    98   7.8  <NA>  1018.8     7     3
sf (17544, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2024-01-01 00:00:00  13.4    82   0.0   360   7.2  <NA>  1018.2     8     4
1 2024-01-01 01:00:00  13.7    79   0.0   360   7.8  <NA>  1018.1     8     3
sd (17544, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2024-01-01 00:00:00  16.1    75   0.0     0   0.0  <NA>  1017.7     6     3
1 2024-01-01 01:00:00  14.4    84   0.0   290   7.6  <NA>  1017.8     5     3
sj (17544, 10)
             datetime  temp  rhum  prcp  wdir  wspd  wpgt    pres  cldc  coco
0 2024-01-01 00:00:00  13.0    77   0.0   320  17.0  <NA>  1018.0     8     3
1 2024-01-01 01:00:00  13.3    77   0.0   320  16.6  <NA>  1017.7     8     3
fres

## Weather Feature Selection

- The Meteostat dataset includes many variables (temp, humidity, precipitation, wind, pressure, cloud cover, etc.).
- For this initial model, we focus on a small set of high-signal features:
  - **temp** → primary driver of electricity demand  
  - **rhum** → affects perceived temperature  
  - **prcp** → captures storm/cloud effects  
  - **wspd** → reflects broader weather patterns  

- This simplification helps to:
  - Focus on the most meaningful drivers of demand  
  - Avoid multicollinearity between overlapping features  
  - Reduce noise and risk of overfitting  
  - Keep the model interpretable and easy to debug  

In [3]:
def prep_weather(df, prefix):
    keep = ["datetime", "temp", "rhum", "prcp", "wspd"]
    keep = [c for c in keep if c in df.columns]
    df = df[keep].copy()
    df = df.rename(columns={c: f"{prefix}_{c}" for c in df.columns if c != "datetime"})
    return df

weather_la = prep_weather(weather_dfs["la"], "la")
weather_sf = prep_weather(weather_dfs["sf"], "sf")
weather_sd = prep_weather(weather_dfs["sd"], "sd")
weather_sj = prep_weather(weather_dfs["sj"], "sj")
weather_fresno = prep_weather(weather_dfs["fresno"], "fresno")

In [4]:
weather_df = weather_la.merge(weather_sf, on="datetime", how="outer")
weather_df = weather_df.merge(weather_sd, on="datetime", how="outer")
weather_df = weather_df.merge(weather_sj, on="datetime", how="outer")
weather_df = weather_df.merge(weather_fresno, on="datetime", how="outer")
weather_df = weather_df.sort_values("datetime").reset_index(drop=True)

print(weather_df.head())
print(weather_df.shape)

             datetime  la_temp  la_rhum  la_prcp  la_wspd  sf_temp  sf_rhum  \
0 2024-01-01 00:00:00     15.9       68      0.0      8.7     13.4       82   
1 2024-01-01 01:00:00     14.2       74      0.0      7.8     13.7       79   
2 2024-01-01 02:00:00     14.1       77      0.0      7.7     12.9       83   
3 2024-01-01 03:00:00     14.0       80      0.0      7.7     12.6       85   
4 2024-01-01 04:00:00     14.0       81      0.0      6.9     11.9       86   

   sf_prcp  sf_wspd  sd_temp  ...  sd_prcp  sd_wspd  sj_temp  sj_rhum  \
0      0.0      7.2     16.1  ...      0.0      0.0     13.0       77   
1      0.0      7.8     14.4  ...      0.0      7.6     13.3       77   
2      0.0      4.4     17.3  ...      0.0      0.0     13.3       75   
3      0.0      1.1     13.3  ...      0.0      0.0     13.3       72   
4      0.0      6.6     12.8  ...      0.0      0.0     13.3       75   

   sj_prcp  sj_wspd  fresno_temp  fresno_rhum  fresno_prcp  fresno_wspd  
0      0.0  

In [5]:
caiso = gridstatus.CAISO()

# monthly chunk boundaries
month_starts = pd.date_range("2024-01-01", "2026-01-01", freq="MS")

dfs = []

for i in range(len(month_starts) - 1):
    start = month_starts[i]
    end = month_starts[i + 1]

    print(f"Loading {start.date()} to {(end - pd.Timedelta(days=1)).date()}")

    df = caiso.get_load(start, end=end, verbose=False)
    dfs.append(df)

# combine into one dataframe
load_5min_df = pd.concat(dfs, ignore_index=True)

print(load_5min_df.shape)
print(load_5min_df.head())
print(load_5min_df.tail())

Loading 2024-01-01 to 2024-01-31


100%|██████████| 31/31 [00:08<00:00,  3.80it/s]


Loading 2024-02-01 to 2024-02-29


100%|██████████| 29/29 [00:07<00:00,  3.76it/s]


Loading 2024-03-01 to 2024-03-31


...

Loading 2025-11-01 to 2025-11-30


100%|██████████| 30/30 [00:07<00:00,  3.79it/s]


Loading 2025-12-01 to 2025-12-31


100%|██████████| 31/31 [00:07<00:00,  3.98it/s]

(210290, 4)
                       Time            Interval Start  \
0 2024-01-01 00:00:00-08:00 2024-01-01 00:00:00-08:00   
1 2024-01-01 00:05:00-08:00 2024-01-01 00:05:00-08:00   
2 2024-01-01 00:10:00-08:00 2024-01-01 00:10:00-08:00   
3 2024-01-01 00:15:00-08:00 2024-01-01 00:15:00-08:00   
4 2024-01-01 00:20:00-08:00 2024-01-01 00:20:00-08:00   

               Interval End     Load  
0 2024-01-01 00:05:00-08:00  21483.0  
1 2024-01-01 00:10:00-08:00  21449.0  
2 2024-01-01 00:15:00-08:00  21350.0  
3 2024-01-01 00:20:00-08:00  21311.0  
4 2024-01-01 00:25:00-08:00  21246.0  
                            Time            Interval Start  \
210285 2025-12-31 23:35:00-08:00 2025-12-31 23:35:00-08:00   
210286 2025-12-31 23:40:00-08:00 2025-12-31 23:40:00-08:00   
210287 2025-12-31 23:45:00-08:00 2025-12-31 23:45:00-08:00   
210288 2025-12-31 23:50:00-08:00 2025-12-31 23:50:00-08:00   
210289 2025-12-31 23:55:00-08:00 2025-12-31 23:55:00-08:00   

                    Interval End     L

In [6]:
load_5min_df["Time"] = pd.to_datetime(load_5min_df["Time"])

load_hourly_df = (
    load_5min_df
    .set_index("Time")["Load"]
    .resample("H")
    .mean()
    .dropna()
    .reset_index()
)

print(load_hourly_df.shape)
print(load_hourly_df.head())
print(load_hourly_df.tail())

(17527, 2)
                       Time          Load
0 2024-01-01 00:00:00-08:00  21199.250000
1 2024-01-01 01:00:00-08:00  20616.833333
2 2024-01-01 02:00:00-08:00  19977.916667
3 2024-01-01 03:00:00-08:00  19550.083333
4 2024-01-01 04:00:00-08:00  19317.750000
                           Time          Load
17522 2025-12-31 19:00:00-08:00  24659.916667
17523 2025-12-31 20:00:00-08:00  23683.916667
17524 2025-12-31 21:00:00-08:00  23008.250000
17525 2025-12-31 22:00:00-08:00  22129.750000
17526 2025-12-31 23:00:00-08:00  21130.583333


/var/folders/r0/mxnnbvln7wl__9w60hwgw01c0000gn/T/ipykernel_81300/3774045327.py:6: FutureWarning: 'H' is deprecated and will be removed in a future version, please use 'h' instead.
  .resample("H")


In [7]:
load_hourly_df.to_csv("../data/raw/caiso_load_2024_2025_hourly.csv", index=False)

In [8]:
print(load_hourly_df.columns.tolist())

['Time', 'Load']


In [9]:
weather_df["datetime"] = pd.to_datetime(weather_df["datetime"])
load_hourly_df["Time"] = pd.to_datetime(load_hourly_df["Time"])

In [10]:
# Remove timezone from load so it matches weather
load_hourly_df["datetime"] = load_hourly_df["Time"].dt.tz_localize(None)

In [11]:
daily_load_df = (
    load_hourly_df
    .set_index("datetime")["Load"]
    .resample("D")
    .mean()
    .reset_index()
    .rename(columns={"Load": "target_load_mw_mean"})
)

print(daily_load_df.shape)
daily_load_df.head()

(731, 2)


,datetime,target_load_mw_mean
0,2024-01-01,20606.559028
1,2024-01-02,23300.965278
2,2024-01-03,23571.142361
3,2024-01-04,23311.829861
4,2024-01-05,23085.718750


In [12]:
daily_load_df = (
    load_hourly_df
    .set_index("datetime")["Load"]
    .resample("D")
    .agg(
        load_mw_mean="mean",
        load_mw_max="max",
        load_mw_min="min",
        load_mw_std="std"
    )
    .reset_index()
)

daily_weather_df = pd.DataFrame(index=weather_df.set_index("datetime").resample("D").size().index)

city_prefixes = ["la", "sf", "sd", "sj", "fresno"]

for city in city_prefixes:
    daily_weather_df[f"{city}_temp_mean"] = weather_df.set_index("datetime")[f"{city}_temp"].resample("D").mean()
    daily_weather_df[f"{city}_temp_max"] = weather_df.set_index("datetime")[f"{city}_temp"].resample("D").max()
    daily_weather_df[f"{city}_temp_min"] = weather_df.set_index("datetime")[f"{city}_temp"].resample("D").min()
    daily_weather_df[f"{city}_rhum_mean"] = weather_df.set_index("datetime")[f"{city}_rhum"].resample("D").mean()
    daily_weather_df[f"{city}_prcp_sum"] = weather_df.set_index("datetime")[f"{city}_prcp"].resample("D").sum()
    daily_weather_df[f"{city}_wspd_mean"] = weather_df.set_index("datetime")[f"{city}_wspd"].resample("D").mean()

daily_weather_df = daily_weather_df.reset_index().rename(columns={"index": "datetime"})

merged_daily_df = daily_weather_df.merge(
    daily_load_df,
    on="datetime",
    how="inner"
)

merged_daily_df["year"] = merged_daily_df["datetime"].dt.year
merged_daily_df["month"] = merged_daily_df["datetime"].dt.month
merged_daily_df["day"] = merged_daily_df["datetime"].dt.day
merged_daily_df["day_of_week"] = merged_daily_df["datetime"].dt.dayofweek
merged_daily_df["day_name"] = merged_daily_df["datetime"].dt.day_name()
merged_daily_df["day_of_year"] = merged_daily_df["datetime"].dt.dayofyear
merged_daily_df["is_weekend"] = (merged_daily_df["day_of_week"] >= 5).astype(int)

print(merged_daily_df.shape)
print(merged_daily_df.head())
print(merged_daily_df.isna().sum().sort_values(ascending=False).head(15))

(731, 42)
    datetime  la_temp_mean  la_temp_max  la_temp_min  la_rhum_mean  \
0 2024-01-01     13.991667         19.0         10.3          68.0   
1 2024-01-02        13.175         18.7          8.2         66.25   
2 2024-01-03     13.670833         17.6         11.1     77.833333   
3 2024-01-04     13.991667         18.0         12.0        45.625   
4 2024-01-05     12.733333         18.7          8.2     50.333333   

   la_prcp_sum  la_wspd_mean  sf_temp_mean  sf_temp_max  sf_temp_min  ...  \
0          0.0          7.75     11.583333         14.4          8.6  ...   
1          0.0         9.825        11.375         16.6          8.2  ...   
2          2.5      9.633333     11.466667         14.4          9.1  ...   
3          0.0         14.45     10.804167         14.8          7.7  ...   
4          0.0          9.05       12.6125         16.4         10.5  ...   

    load_mw_max   load_mw_min  load_mw_std  year  month  day  day_of_week  \
0  24801.833333  16828.250000

In [13]:
print(merged_daily_df.shape)
print(merged_daily_df.columns.tolist())

(731, 42)
['datetime', 'la_temp_mean', 'la_temp_max', 'la_temp_min', 'la_rhum_mean', 'la_prcp_sum', 'la_wspd_mean', 'sf_temp_mean', 'sf_temp_max', 'sf_temp_min', 'sf_rhum_mean', 'sf_prcp_sum', 'sf_wspd_mean', 'sd_temp_mean', 'sd_temp_max', 'sd_temp_min', 'sd_rhum_mean', 'sd_prcp_sum', 'sd_wspd_mean', 'sj_temp_mean', 'sj_temp_max', 'sj_temp_min', 'sj_rhum_mean', 'sj_prcp_sum', 'sj_wspd_mean', 'fresno_temp_mean', 'fresno_temp_max', 'fresno_temp_min', 'fresno_rhum_mean', 'fresno_prcp_sum', 'fresno_wspd_mean', 'load_mw_mean', 'load_mw_max', 'load_mw_min', 'load_mw_std', 'year', 'month', 'day', 'day_of_week', 'day_name', 'day_of_year', 'is_weekend']


In [14]:
print(merged_daily_df["datetime"].min(), merged_daily_df["datetime"].max())
print(merged_daily_df.isna().sum().sort_values(ascending=False).head(15))

2024-01-01 00:00:00 2025-12-31 00:00:00
datetime            0
load_mw_mean        0
sj_prcp_sum         0
sj_wspd_mean        0
fresno_temp_mean    0
fresno_temp_max     0
fresno_temp_min     0
fresno_rhum_mean    0
fresno_prcp_sum     0
fresno_wspd_mean    0
load_mw_max         0
la_temp_mean        0
load_mw_min         0
load_mw_std         0
year                0
dtype: int64


In [15]:
merged_daily_df["target_load_mw_mean"] = merged_daily_df["load_mw_mean"]

In [16]:
merged_daily_df.to_csv("../data/raw/model_input_daily_2024_2025.csv", index=False)

## Data Wrangling Summary

- Collected hourly weather data for 2024 and 2025 from 5 major CA cities: LA, SF, SD, SJ, and Fresno. We kept only key features: temp, humidity, precipitation, wind.
- Cleaned and combined 2 years of CAISO load data; built a proper hourly `datetime` column.
- Merged weather + load into a single hourly dataset (~17,527 rows) with no missing data.

- Aggregated to daily level using summary stats (mean, max, min, std).
- Created target variable: daily average load (`target_load_mw_mean`).
- Added calendar features (month, day_of_week, day_of_year, weekend flag).

- Final outputs:
  - `caiso_load_2024_2025_hourly.csv` (hourly data for EDA purposes)
  - `model_input_daily_2024_2025.csv` (model-ready dataset)

**Takeaway:**  
Clean, aligned dataset combining statewide energy demand with multi-city weather + time features, ready for modeling daily CA load.